# PesoVision — EDA (USD/MXN)

Exploración de calidad de datos y distribución del target para modelado.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
features_path = ROOT / "data" / "processed" / "fx_features.csv"
if not features_path.exists():
    features_path = ROOT / "data" / "processed" / "fx_features.parquet"
    df = pd.read_parquet(features_path)
else:
    df = pd.read_csv(features_path)

df["date"] = pd.to_datetime(df["date"])
df.head()

## 1. Serie temporal del tipo de cambio

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["date"], df["close"], color="#1f77b4")
ax.set_title("USD/MXN — precio de cierre")
ax.set_xlabel("Fecha")
ax.set_ylabel("Close")
plt.tight_layout()
plt.show()

## 2. Distribución de retornos diarios

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["return_1d"].dropna(), bins=50, color="#2ca02c", edgecolor="white")
ax.set_title("Distribución de return_1d")
ax.set_xlabel("Retorno diario")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

## 3. Balance del target (direction_next_day)

In [ ]:
target_counts = df["direction_next_day"].value_counts().sort_index()
pct_up = target_counts.get(1, 0) / len(df) * 100

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind="bar", ax=ax, color=["#d62728", "#1f77b4"])
ax.set_title("Balance del target: 0=Baja, 1=Sube")
ax.set_xlabel("Clase")
ax.set_ylabel("Conteo")
plt.tight_layout()
plt.show()

print(f"Observaciones: {len(df)}")
print(f"Rango: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"% subidas (target=1): {pct_up:.1f}%")

## Conclusiones

- **Cobertura temporal:** la serie cubre varios años de datos diarios USD/MXN desde 2019.
- **Calidad:** el ETL elimina nulos críticos, duplicados y retornos diarios > 15% (outliers).
- **Target:** `direction_next_day` está relativamente balanceado (~50/50), adecuado para clasificación binaria.
- **Retornos:** distribución centrada cerca de 0 con colas moderadas; coherente con un mercado FX diario.
- **Modelado:** features técnicas (retornos, medias móviles, volatilidad) están listas en `fx_features`.